# yolov5n-voc sur voc — entraînement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/EmbeddedComputervision/blob/main/notebooks/voc/yolov5n-voc_train.ipynb)

Rôle : entraînement ; palier **N** (`EPOCHS` epochs of `tools/train_v5.py`, days on CPU: [long runs](../../docs/tasks/long-runs.md)) ; évaluation finale sur `SUBSET` images. Tâches : [T33.7](../../docs/tasks/M33-yolov5n.md) ;
notebooks : [M14](../../docs/tasks/M14-notebooks.md).

Prérequis :
- données : `data/VOCdevkit/VOC2007/ImageSets/Main/test.txt` — tools/get_voc.sh
- poids : `weights/ultralytics/yolov5n.pt` — tools/get_weights.sh

Chaque étape affiche la commande `python tools/…` qu'elle lance : un résultat se
reproduit en ligne de commande. Sorties dans `build/notebooks/voc/yolov5n-voc/`, jamais dans `results/`
(une mAP sur `SUBSET` images n'est pas publiable, règles de M12). Notebook généré par
`python -m tools.notebooks` (`make notebooks`) : modifier le gabarit
`tools/notebooks/gabarits.py`, pas ce fichier.

## Paramètres

In [ ]:
NET          = 'yolov5n-voc'  # nom de CFG_FILES ou cfg (relative à la racine)
DATASET      = 'voc'  # clé de DATASETS
SPLIT        = None  # None : split d'évaluation du jeu
WEIGHTS      = 'build/notebooks/voc/yolov5n-voc/final.npz'  # EMA weights, evaluated at the end
DEVICE       = 'gpu'  # cpu | gpu (entraînement seulement ; le reste sur CPU)
SUBSET       = 50  # images évaluées ; 0 : split complet (palier N)
SIZE         = None  # entrée LxH (ex. '640x192') ; None : celle de la cfg
RESIZE       = 'v5'  # v5 (letterbox and thresholds of YOLOv5, T33.4)
METRIC       = None  # None : celle du jeu (coco pour coco et flir)
DATA_ROOT    = None  # racine du jeu ; None : data/<jeu> (Colab : dossier Drive)
INIT         = 'coco'  # coco (heads reinitialized) | he
EPOCHS       = 30  # epochs of the recipe (warmup 3, linear lr)
ITERS        = 0  # stop after n batches (0: all epochs; a short try: 20)
BATCH        = 16  # batch (gradients accumulated to 64)
LR           = 0.01  # lr0 (final lr: 0.01·lr0)
SIZE_TRAIN   = 640  # square training input
TRAIN_SUBSET = 0  # first n training images (0: all)
WORKERS      = 4
DRIVE_DIR    = '/content/drive/MyDrive/EmbeddedCV'  # Colab : archives data/<jeu>.tar, sorties et reprise dans runs/ ; None : pas de Drive
JOBS         = None  # processus des évaluations ; None : un par cœur CPU, plafonné par la mémoire (runtime TPU de Colab : des dizaines)
SHOW         = 6  # images affichées
OUT          = 'build/notebooks/voc/yolov5n-voc'  # sorties du notebook
REPO_URL     = 'https://github.com/lrivals/EmbeddedComputervision.git'  # Colab : dépôt cloné
REV          = 'main'  # Colab : révision

## Environnement

Local : rien n'est installé ni téléchargé, la cellule vérifie les prérequis et
s'arrête avec la commande à lancer. Colab : clone du dépôt, installation, CuPy si
`DEVICE = "gpu"`, poids, puis données par `colab.prepare_data` (archive
`<DRIVE_DIR>/data/<jeu>.tar` ou téléchargement direct ; ExDark et FLIR : archive
seulement, faite sur le PC par `tools/get_datasets.sh pack <jeu>`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COLAB = "google.colab" in sys.modules


def repo_root():
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "tools" / "notebooks" / "commandes.py").exists():
            return d
    return None


ROOT = repo_root()
if COLAB:
    if ROOT is None:
        ROOT = Path("/content/EmbeddedComputervision")
        if not ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    # Clone d'une session précédente : remis à REV (notebook et code du même commit).
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "FETCH_HEAD"], check=True)
    for m in [m for m in sys.modules if m.split(".")[0] in ("tools", "yolo")]:
        del sys.modules[m]  # modules importés avant la mise à jour
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}/python[data,plots]"], check=True)
    if DEVICE == "gpu":
        # Runtime CPU : pas de pilote, CuPy dirait « cudaErrorInsufficientDriver ».
        try:
            smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        except FileNotFoundError:
            smi = None
        if smi is None or smi.returncode != 0:
            raise RuntimeError("runtime Colab sans GPU : créer un nouveau serveur Colab "
                               "de type GPU (T4, L4 ou A100), ou DEVICE = 'cpu'")
        cuda = smi.stdout.split("CUDA Version:")[-1].split()[0]  # version du pilote
        cupy_pkg = "cupy-cuda13x" if int(cuda.split(".")[0]) >= 13 else "cupy-cuda12x"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", cupy_pkg],
                       check=True)
elif ROOT is None:
    raise RuntimeError("racine du dépôt introuvable : ouvrir le notebook depuis notebooks/")
os.chdir(ROOT)
for p in (ROOT, ROOT / "python"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tools.notebooks import commandes as C
from tools.notebooks import env

TRACE = env.trace(DEVICE)
env.check_device(DEVICE)
env.prepare(DATASET, 'weights/ultralytics/yolov5n.pt' if INIT == 'coco' else (), None, DATA_ROOT, COLAB, hint='',
            drive_dir=DRIVE_DIR)
Path(OUT).mkdir(parents=True, exist_ok=True)

## Fine-tuning

`tools/train_v5.py` in a subprocess: SGD Nesterov with three parameter groups,
warmup then linear lr, gradients accumulated to a nominal batch of 64, weight EMA,
mosaic / random perspective / HSV / flip, heads reinitialized from the COCO weights
with the bias prior of the reference (YOLOv7-tiny: the OTA loss of
`hyp.scratch.tiny.yaml`). A `checkpoint.npz` is written every 200 batches; running
the cell again resumes (`--resume`).

In [ ]:
import csv

from IPython.display import Markdown, display

RUN = Path(OUT)
RUN.mkdir(parents=True, exist_ok=True)
resume = (RUN / "checkpoint.npz").exists()
C.run(C.cmd_train_v5(NET, RUN, DATASET, INIT, resume, EPOCHS, ITERS, BATCH, SIZE_TRAIN,
                     WORKERS, DEVICE, DATA_ROOT, TRAIN_SUBSET, LR), log=RUN / "log.txt")

## Losses

Last rows of `loss.csv` (per batch: box, objectness and class terms per image).

In [ ]:
rows = list(csv.DictReader((RUN / "loss.csv").open())) if (RUN / "loss.csv").exists() else []
for r in rows[-10:]:
    print(", ".join(f"{k} {r[k]}" for k in ("it", "lr", "loss", "box", "obj", "cls")))

## Evaluation

The EMA weights by `tools/eval_voc.py --preset v5` (T33.4) on `SUBSET` images.

In [ ]:
result = RUN / "map_float.md"
result.unlink(missing_ok=True)
C.run(C.cmd_eval_voc(NET, WEIGHTS, DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE,
                     DATA_ROOT, out=RUN / "dets", markdown=result, jobs=JOBS))
display(Markdown(result.read_text()))

## Résumé

In [ ]:
print(f"outputs: {RUN}/ (checkpoint.npz, loss.csv, final weights) ; révision {TRACE['rev']}")
print("commandes équivalentes :")
for c in C.HISTORY:
    print("  " + c)